# 🛰️ EuroSAT RGB — Land Use & Land Cover Classification

| | |
|---|---|
| **Dataset** | EuroSAT RGB (Sentinel-2 Satellite Imagery) |
| **Task** | Multi-class Image Classification (10 classes) |
| **Model** | Transfer Learning — ResNet50 (ImageNet pretrained) |
| **Framework** | PyTorch |
| **Python** | 3.10.19 |

---

### 🗂️ 10 Land Cover Classes
`AnnualCrop` · `Forest` · `HerbaceousVegetation` · `Highway` · `Industrial`  
`Pasture` · `PermanentCrop` · `Residential` · `River` · `SeaLake`

---
### 📋 Notebook Sections
1. Imports & Environment Setup  
2. Hyperparameters  
3. Data Transforms & Augmentation  
4. Load Dataset & Train/Val/Test Split  
5. Exploratory Data Analysis  
6. ResNet50 Model Architecture  
7. Loss, Optimizer & Scheduler  
8. Training Loop  
9. Training Curves  
10. Test Evaluation  
11. Confusion Matrix & Classification Report  
12. Per-Class Accuracy  
13. Save Model Checkpoint  

## 1️⃣ Imports & Environment Setup

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter
from tqdm.notebook import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Determine device and verify CUDA support.
DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    try:
        torch.tensor([0.0], device='cuda')
        DEVICE = torch.device('cuda')
    except Exception as e:
        print('WARNING: CUDA is available but not usable in this PyTorch build.')
        print(f'         Falling back to CPU. Error: {e}')

print('=' * 50)
print(f'  PyTorch  : {torch.__version__}')
print(f'  Device   : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU      : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('=' * 50)

## 2️⃣ Hyperparameters

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
# If cloning this repo, ensure the dataset is extracted to 'data/Eurosat_Dataset'
DATA_DIR    = 'data/Eurosat_Dataset'   # Root folder with class subfolders
NUM_CLASSES = 10
IMG_SIZE    = 224                 # ResNet50 standard input size

# ── Output folder (all results saved here) ───────────────────────────────────
RESULTS_DIR = 'results/resnet50'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE  = 32               # Reduced from 64 — ResNet50 uses more VRAM
NUM_EPOCHS  = 20
LR          = 1e-4             # Lower LR for fine-tuning pretrained weights
WEIGHT_DECAY= 1e-4

# ── Split ratio ───────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO  = 0.15  (remainder)

print(f'  Data dir    : {DATA_DIR}')
print(f'  Results dir : {RESULTS_DIR}')
print(f'  Image size  : {IMG_SIZE}×{IMG_SIZE}  (ResNet50 standard)')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Epochs      : {NUM_EPOCHS}')
print(f'  LR          : {LR}')
print(f'  Split       : {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/15')

## 3️⃣ Data Transforms & Augmentation

> **EuroSAT RGB channel statistics** (pre-computed from the full dataset):
> - Mean: `[0.3444, 0.3803, 0.4078]`
> - Std:  `[0.2034, 0.1365, 0.1148]`

> **Note:** Images are resized to **224×224** to match ResNet50's expected input.

Training set uses augmentation; validation and test sets use only resize + normalize.

In [ ]:
MEAN = [0.3444, 0.3803, 0.4078]
STD  = [0.2034, 0.1365, 0.1148]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

print('Train transforms  :\n', train_transform)
print('\nVal/Test transforms:\n', val_test_transform)

## 4️⃣ Load Dataset & Train / Val / Test Split

| Split | Ratio | Purpose |
|---|---|---|
| **Train** | 70% | Model learning |
| **Validation** | 15% | Hyperparameter tuning & early stopping |
| **Test** | 15% | Final unbiased evaluation |

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class FlatImageDataset(Dataset):
    """Loads flat folder of images, extracts class from filename prefix."""
    
    def __init__(self, data_dir, class_names, transform=None):
        self.transform   = transform
        self.class_names = class_names
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        
        self.samples = []  # (filepath, label)
        
        for fname in sorted(os.listdir(data_dir)):
            if not fname.lower().endswith('.jpg'):
                continue
            # AnnualCrop_1.jpg → AnnualCrop
            prefix = '_'.join(fname.replace('.jpg','').split('_')[:-1])
            if prefix in self.class_to_idx:
                path  = os.path.join(data_dir, fname)
                label = self.class_to_idx[prefix]
                self.samples.append((path, label))
        
        self.targets = [s[1] for s in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation",
    "Highway", "Industrial", "Pasture",
    "PermanentCrop", "Residential", "River", "SeaLake"
]

full_train_ds = FlatImageDataset(DATA_DIR, CLASS_NAMES, transform=train_transform)
full_eval_ds  = FlatImageDataset(DATA_DIR, CLASS_NAMES, transform=val_test_transform)

total   = len(full_train_ds)
n_train = int(TRAIN_RATIO * total)
n_val   = int(VAL_RATIO   * total)
n_test  = total - n_train - n_val

all_indices = list(range(total))
rng = np.random.RandomState(SEED)
rng.shuffle(all_indices)

train_idx = all_indices[:n_train]
val_idx   = all_indices[n_train : n_train + n_val]
test_idx  = all_indices[n_train + n_val:]

train_set = Subset(full_train_ds, train_idx)
val_set   = Subset(full_eval_ds,  val_idx)
test_set  = Subset(full_eval_ds,  test_idx)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=(DEVICE.type=='cuda'))
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE.type=='cuda'))

print(f'Total images : {total:,}')
print(f'Train        : {n_train:,}')
print(f'Val          : {n_val:,}')
print(f'Test         : {n_test:,}')
print(f'Classes      : {CLASS_NAMES}')

## 5️⃣ Exploratory Data Analysis

### 5a. One Sample Image per Class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('EuroSAT RGB — One Sample per Class', fontsize=14, fontweight='bold')

shown, ax_iter = set(), iter(axes.flat)
for idx in range(len(full_eval_ds)):
    img_t, label = full_eval_ds[idx]
    if label not in shown:
        ax = next(ax_iter)
        # Denormalize for display
        img_np = img_t.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(STD) + np.array(MEAN)
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(CLASS_NAMES[label], fontsize=9)
        ax.axis('off')
        shown.add(label)
    if len(shown) == NUM_CLASSES:
        break

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'eda_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/eda_samples.png')

### 5b. Class Distribution

In [ ]:
label_counts = Counter(full_train_ds.targets)
counts = [label_counts[i] for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(CLASS_NAMES, counts, color='steelblue', edgecolor='white')
ax.set_title('Class Distribution — Full Dataset', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Images')
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.yaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(cnt), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'eda_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/eda_distribution.png')

## 6️⃣ ResNet50 Model Architecture

| Component | Detail |
|---|---|
| **Backbone** | ResNet50 pretrained on ImageNet |
| **Strategy** | Full fine-tuning (all layers trainable) |
| **Head** | Original FC (2048→1000) replaced with FC (2048→10) |
| **Weights** | Loaded from local `.pth` file (no internet needed) |

> 💡 **Download weights once** from `https://download.pytorch.org/models/resnet50-11ad3fa6.pth`  
> and save to `C:/weights/resnet50-11ad3fa6.pth` (or update `WEIGHTS_PATH` below).

In [ ]:
# ── Weights path — update this to wherever you saved the .pth file ────────────
WEIGHTS_PATH = r'C:/weights/resnet50-11ad3fa6.pth'

# ── Load ResNet50 with fallback chain ─────────────────────────────────────────
if os.path.exists(WEIGHTS_PATH):
    model = models.resnet50(weights=None)
    state_dict = torch.load(WEIGHTS_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print('✅ Loaded pretrained ImageNet weights from local file.')

elif os.path.isdir(r'C:/my_torch_cache'):
    import os as _os
    _os.environ['TORCH_HOME'] = r'C:/my_torch_cache'
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    print('✅ Loaded pretrained weights from custom torch cache.')

else:
    model = models.resnet50(weights=None)
    print('⚠️  No pretrained weights found — training from scratch.')
    print('   Download weights from: https://download.pytorch.org/models/resnet50-11ad3fa6.pth')
    print(f'   Then save to: {WEIGHTS_PATH}')

# ── Replace final FC layer for 10-class EuroSAT ───────────────────────────────
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# ── Parameter summary ─────────────────────────────────────────────────────────
total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters     : {total_p:,}')
print(f'Trainable parameters : {trainable_p:,}')
print(f'FC layer replaced    : 2048 → {NUM_CLASSES} classes')

## 7️⃣ Loss Function, Optimizer & Scheduler

| Component | Choice | Reason |
|---|---|---|
| **Loss** | CrossEntropyLoss | Standard for multi-class classification |
| **Optimizer** | Adam (lr=1e-4) | Lower LR for stable fine-tuning of pretrained weights |
| **Scheduler** | StepLR (γ=0.5, step=5) | Halves LR every 5 epochs |

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f'Loss      : CrossEntropyLoss')
print(f'Optimizer : Adam  (lr={LR}, weight_decay={WEIGHT_DECAY})')
print(f'Scheduler : StepLR (step_size=5, gamma=0.5)')

## 8️⃣ Training Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, phase='train'):
    """Single epoch of training or evaluation."""
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, desc=f'{phase.capitalize():5s}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total


print('Training functions ready.')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, 'best_model.pth')

for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc='Epochs'):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, phase='train')
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion, phase='val')
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    flag = ' ✅ (best)' if vl_acc > best_val_acc else ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS}]  '
          f'Train  loss={tr_loss:.4f}  acc={tr_acc*100:.2f}%  |  '
          f'Val  loss={vl_loss:.4f}  acc={vl_acc*100:.2f}%{flag}')

print(f'\nBest Validation Accuracy : {best_val_acc*100:.2f}%')
print(f'Best model saved → {BEST_MODEL_PATH}')

## 9️⃣ Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EuroSAT ResNet50 — Training Curves', fontsize=13, fontweight='bold')

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', markersize=3, label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   'r--s',markersize=3, label='Val Loss',   linewidth=2)
axes[0].set_title('Loss per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', markersize=3, label='Train Acc', linewidth=2)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r--s',markersize=3, label='Val Acc',   linewidth=2)
axes[1].set_title('Accuracy per Epoch'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/training_curves.png')

## 🔟 Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))

test_loss, test_acc = run_epoch(model, test_loader, criterion, phase='test')

print('=' * 40)
print(f'  Test Loss     : {test_loss:.4f}')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print('=' * 40)

## 1️⃣1️⃣ Confusion Matrix & Classification Report

In [ ]:
def get_all_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Predicting', leave=False):
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())
    return np.array(y_true), np.array(y_pred)


y_true, y_pred = get_all_predictions(model, test_loader)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label',      fontsize=12)
plt.title('Confusion Matrix — ResNet50 on EuroSAT RGB Test Set', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/confusion_matrix.png')

print('\n' + '='*65)
print('Classification Report')
print('='*65)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

## 1️⃣2️⃣ Per-Class Accuracy

In [ ]:
per_class_correct = np.zeros(NUM_CLASSES)
per_class_total   = np.zeros(NUM_CLASSES)

for t, p in zip(y_true, y_pred):
    per_class_total[t]   += 1
    if t == p:
        per_class_correct[t] += 1

per_class_acc = per_class_correct / per_class_total * 100

colors_acc = ['#2ecc71' if a >= 90 else '#e67e22' if a >= 75 else '#e74c3c' for a in per_class_acc]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(CLASS_NAMES, per_class_acc, color=colors_acc, edgecolor='white')
ax.set_xlabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy on Test Set — ResNet50', fontsize=13, fontweight='bold')
ax.set_xlim(0, 105)
ax.xaxis.grid(True, alpha=0.3); ax.set_axisbelow(True)
ax.axvline(x=90, color='gray', linestyle='--', alpha=0.4, label='90% threshold')
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{acc:.1f}%', va='center', fontsize=9)
patches = [mpatches.Patch(color='#2ecc71', label='≥ 90%'),
           mpatches.Patch(color='#e67e22', label='75–89%'),
           mpatches.Patch(color='#e74c3c', label='< 75%')]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'per_class_accuracy.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nPer-class accuracy:')
for c, acc in zip(CLASS_NAMES, per_class_acc):
    print(f'  {c:<25} {acc:.2f}%')

## 1️⃣3️⃣ Save Model Checkpoint

In [ ]:
checkpoint = {
    'architecture'   : 'ResNet50',
    'num_classes'    : NUM_CLASSES,
    'class_names'    : CLASS_NAMES,
    'img_size'       : IMG_SIZE,
    'epochs_trained' : NUM_EPOCHS,
    'best_val_acc'   : best_val_acc,
    'test_acc'       : test_acc,
    'model_state'    : model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'hyperparams'    : {
        'batch_size': BATCH_SIZE, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'seed': SEED
    }
}

CHECKPOINT_PATH = os.path.join(RESULTS_DIR, 'eurosat_resnet50_final.pth')
torch.save(checkpoint, CHECKPOINT_PATH)
print(f'Checkpoint saved → {CHECKPOINT_PATH}')
print(f'Best Val Accuracy : {best_val_acc*100:.2f}%')
print(f'Test Accuracy     : {test_acc*100:.2f}%')

print('\nOutputs produced in', RESULTS_DIR, ':')
expected_files = [
    'eda_samples.png', 'eda_distribution.png', 'training_curves.png',
    'confusion_matrix.png', 'per_class_accuracy.png',
    'best_model.pth', 'eurosat_resnet50_final.pth'
]
for f in expected_files:
    full_path = os.path.join(RESULTS_DIR, f)
    status = '✅' if os.path.exists(full_path) else '❌ (not yet generated)'
    print(f'  {status}  {full_path}')